# V5: missing-input estimates and typical building load

`candidate_v4.csv` scored **3.37** in the competition. V5 blends that exact file 50/50 with a new model estimating missing inputs and typical load from observed train/test inputs.

The second-round weighted holdout check improved. Ordinary and missing-pattern-only scores were nearly unchanged. V5's competition score is not yet known. See `model_v5_review.md` for details.


In [ ]:
import json
import pandas as pd
from energy_model import ROOT
from energy_model_v5 import write_submission
results = json.loads((ROOT / 'validation_v5_results.json').read_text())


## Compare the second-round holdout

These 1,600 rows were excluded from second-round selection; some were used in first-round experiments. Mask trials share rows. The weighted check approximates the test building/event mix and is not an official competition score.


In [ ]:
pd.DataFrame({
    'Current 3.37 model': {
        'Ordinary holdout': results['baseline']['ordinary']['rmse'],
        'Test missing patterns': results['baseline']['test_masks_rmse'],
        'Building/event weighted': results['baseline']['weighted_rmse'],
    },
    'V5': {
        'Ordinary holdout': results['candidate']['ordinary']['rmse'],
        'Test missing patterns': results['candidate']['test_masks_rmse'],
        'Building/event weighted': results['candidate']['weighted_rmse'],
    },
}).round(4)


## Regenerate the submission

Use the project's existing Python environment. CatBoost is installed in `.venv`. The next cell fits on all 8,000 labelled rows, learns input-value relationships from train/test covariates, and blends with the checked `candidate_v4.csv`. It writes `candidate_v5.csv`.


In [ ]:
submission = write_submission()
test = pd.read_csv(ROOT / 'test.csv')
assert len(submission) == 3000
assert submission['id'].equals(test['id'])
submission.head()


## Reproduce validation if needed

This keeps selected settings fixed, refits on the 6,400 development rows, and rewrites `validation_v5_results.json`. It does not tune on holdout scores.


In [ ]:
from validate_energy_model_v5 import validate
reproduced = validate()
